# Mechanistic Analysis v2 — Both Models

Subjects: seed-0 best checkpoints, Study 1, both models (`results_v2/{transformer,lstm}/study1/seed0/best_model.pt`).
Fingerprints asserted on load (Transformer val=7.5%, LSTM val=35.1%) — no unverified model enters analysis.

Population: `datasets_v2/study1/` val + all four OOD bucket files (ops4-7).

**Protocol**: all decoding below is genuinely autoregressive (free-running greedy), matching `main.py::evaluate_model()` and verified identical to the reported training-time numbers in `cold_verify.py`. Any probe that structurally requires teacher forcing will print an explicit protocol banner at the point it's used (Phase 3).

This notebook does **not** modify `analysis_plots.ipynb`, which documents the legacy (pre-fix) analysis.

## Phase 0 — Setup and subject selection

In [ ]:
import sys
import json
from pathlib import Path
from collections import defaultdict, Counter

sys.path.append("../../")
import torch

from data.tokenizer import create_tokenizer
from models.transformer import create_transformer_model
from models.lstm import create_lstm_model

MAX_INPUT_LEN = 64
MAX_OUTPUT_LEN = 12
DATA_DIR = Path("../../datasets_v2/study1")
FILES = {
    "val": "val.json",
    "ops4": "ood_ops4.json",
    "ops5": "ood_ops5.json",
    "ops6": "ood_ops6.json",
    "ops7": "ood_ops7.json",
}

tokenizer = create_tokenizer()
pad_idx = tokenizer.pad_idx

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_device()
print(f"Device: {DEVICE}")

In [ ]:
# Load + assert fingerprints - no unverified model enters analysis, ever.

def load_transformer():
    ckpt = torch.load("../../results_v2/transformer/study1/seed0/best_model.pt", map_location="cpu")
    assert ckpt["val_accuracy"] == 7.5, f"TRANSFORMER FINGERPRINT MISMATCH: val_accuracy={ckpt['val_accuracy']} (expected 7.5)"
    model = create_transformer_model(
        vocab_size=tokenizer.vocab_size, d_model=256, nhead=8,
        num_encoder_layers=3, num_decoder_layers=3, pad_idx=pad_idx,
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(DEVICE)
    model.eval()
    print(f"Transformer loaded. Fingerprint OK: epoch={ckpt['epoch']}, val_accuracy={ckpt['val_accuracy']}, "
          f"seed={ckpt['seed']}, generator_commit={ckpt['generator_commit']}")
    return model

def load_lstm():
    ckpt = torch.load("../../results_v2/lstm/study1/seed0/best_model.pt", map_location="cpu")
    assert ckpt["val_accuracy"] == 35.1, f"LSTM FINGERPRINT MISMATCH: val_accuracy={ckpt['val_accuracy']} (expected 35.1)"
    model = create_lstm_model(vocab_size=tokenizer.vocab_size, embedding_dim=128, hidden_size=256)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(DEVICE)
    model.eval()
    print(f"LSTM loaded. Fingerprint OK: epoch={ckpt['epoch']}, val_accuracy={ckpt['val_accuracy']}, "
          f"seed={ckpt['seed']}, generator_commit={ckpt['generator_commit']}")
    return model

transformer = load_transformer()
lstm = load_lstm()

In [ ]:
def encode_checked(text, max_len):
    ids = tokenizer.encode(text)
    if len(ids) > max_len:
        raise ValueError(f"too long: {text!r} ({len(ids)} > {max_len})")
    return ids

def encode_input(expr):
    ids = encode_checked(expr, MAX_INPUT_LEN)
    ids = ids + [pad_idx] * (MAX_INPUT_LEN - len(ids))
    return torch.tensor(ids, dtype=torch.long)

def detect_op_type(expr_str):
    counts = {"+": 0, "-": 0, "*": 0, "/": 0}
    for ch in expr_str:
        if ch in counts:
            counts[ch] += 1
    return max(counts, key=counts.get) if any(counts.values()) else "mixed"

def autoregressive_decode(model, src, device, max_len=MAX_OUTPUT_LEN):
    """Genuine free-running greedy decode - PROTOCOL: autoregressive, no teacher
    forcing. Recomputes the full decoder input each step (no KV cache in either
    model's forward), which is correctness-equivalent to incremental decoding for
    both the Transformer and the LSTM. Matches main.py::evaluate_model()."""
    model.eval()
    with torch.no_grad():
        src_in = src.unsqueeze(0).to(device)
        dec_ids = [tokenizer.sos_idx]
        for _ in range(max_len - 1):
            cur = dec_ids + [pad_idx] * (max_len - len(dec_ids))
            cur = cur[:max_len]
            dec_tensor = torch.tensor([cur], dtype=torch.long).to(device)
            out = model(src_in, dec_tensor)
            nxt = out[0, len(dec_ids) - 1, :].argmax(dim=-1).item()
            if nxt == tokenizer.eos_idx or nxt == pad_idx:
                break
            dec_ids.append(nxt)
    return dec_ids[1:]

def first_error_step(pred_ids, target_str):
    """Compare predicted token stream against the ground-truth answer tokens +
    EOS, position by position. Returns the 0-indexed step of the first
    mismatch, or None if the full (answer+EOS) sequence matches."""
    target_ids = tokenizer.encode(target_str) + [tokenizer.eos_idx]
    pred_ids_eos = pred_ids + [tokenizer.eos_idx] if (len(pred_ids) == 0 or pred_ids[-1] != tokenizer.eos_idx) else pred_ids
    for i, gt in enumerate(target_ids):
        pt = pred_ids_eos[i] if i < len(pred_ids_eos) else None
        if pt != gt:
            return i
    return None

def classify_error(pred_str, target_str):
    """sign-error: exact magnitude match, flipped sign.
    magnitude-ballpark: same sign, ratio within 2x (or target=0 and |pred|<=2).
    format-garbage: doesn't parse as an integer at all.
    other: everything else."""
    try:
        pred_val = int(pred_str)
    except (ValueError, TypeError):
        return "format-garbage"
    target_val = int(target_str)
    if pred_val == target_val:
        return "correct"
    if target_val == 0:
        return "magnitude-ballpark" if abs(pred_val) <= 2 else "other"
    if pred_val == -target_val:
        return "sign-error"
    same_sign = (pred_val > 0) == (target_val > 0)
    ratio = abs(pred_val) / abs(target_val)
    if same_sign and 0.5 <= ratio <= 2.0:
        return "magnitude-ballpark"
    return "other"

In [ ]:
def run_model_over_file(model, model_name, file_key, filename):
    data = json.loads((DATA_DIR / filename).read_text())["data"]
    results = []
    for item in data:
        expr = item["input"]
        target = str(item["output"])
        op = detect_op_type(expr)
        src = encode_input(expr)
        pred_ids = autoregressive_decode(model, src, DEVICE)
        pred_str = tokenizer.decode(pred_ids)
        correct = pred_str == target
        fe_step = first_error_step(pred_ids, target) if not correct else None
        err_class = None if correct else classify_error(pred_str, target)
        results.append({
            "expression": expr, "target": target, "pred": pred_str,
            "op": op, "num_operations": item["num_operations"],
            "correct": correct, "first_error_step": fe_step,
            "error_class": err_class,
        })
    print(f"  [{model_name}] {file_key} ({filename}): {len(results)} examples, "
          f"{sum(r['correct'] for r in results)} correct")
    return results

all_results = {"transformer": {}, "lstm": {}}
for model_name, model in [("transformer", transformer), ("lstm", lstm)]:
    print(f"\n=== Running {model_name} over all files ===")
    for file_key, filename in FILES.items():
        all_results[model_name][file_key] = run_model_over_file(model, model_name, file_key, filename)

## Phase 1 — Behavioral (both models, identical treatment)

### 1.1 Per-operation accuracy (CP4 successor)
Correct/total per operator, val + each OOD bucket, both models. Division counts are thin (~4.5% of data) — n is reported alongside every rate.

In [ ]:
op_accuracy = {}
for model_name in ["transformer", "lstm"]:
    op_accuracy[model_name] = {}
    for file_key in FILES:
        recs = all_results[model_name][file_key]
        by_op = defaultdict(lambda: [0, 0])
        for r in recs:
            by_op[r["op"]][1] += 1
            if r["correct"]:
                by_op[r["op"]][0] += 1
        op_accuracy[model_name][file_key] = {
            op: {"correct": c, "total": t, "accuracy": round(100.0 * c / t, 2) if t else None}
            for op, (c, t) in by_op.items()
        }
        print(f"\n[{model_name}] {file_key}:")
        for op in ["+", "-", "*", "/"]:
            if op in by_op:
                c, t = by_op[op]
                print(f"  {op}: {c}/{t} = {100.0*c/t:.2f}%")

### 1.2 Failure onset (CP5 successor)
First-error-step distribution over ALL val+OOD failures, not a 5-trace sample — full histograms per model. The n=5 caveat from the legacy analysis ends here.

In [ ]:
failure_histograms = {}
for model_name in ["transformer", "lstm"]:
    all_failures_steps = []
    for file_key in FILES:
        for r in all_results[model_name][file_key]:
            if not r["correct"]:
                all_failures_steps.append(r["first_error_step"])
    hist = Counter(all_failures_steps)
    failure_histograms[model_name] = dict(sorted(hist.items()))
    total_failures = len(all_failures_steps)
    print(f"\n[{model_name}] total failures: {total_failures}")
    for step in sorted(hist.keys()):
        pct = 100.0 * hist[step] / total_failures
        print(f"  step {step}: {hist[step]} ({pct:.1f}%)")

### 1.3 Error taxonomy (new, motivated by the cold-verify decode)
Classify errors as sign-error / magnitude-ballpark (within 2x) / format-garbage / other. Hypothesis on record: LSTM errors cluster near-miss, Transformer errors are calibrated-format-random-value.

In [ ]:
taxonomy = {}
for model_name in ["transformer", "lstm"]:
    counts = Counter()
    for file_key in FILES:
        for r in all_results[model_name][file_key]:
            if not r["correct"]:
                counts[r["error_class"]] += 1
    total = sum(counts.values())
    taxonomy[model_name] = {k: {"count": v, "pct": round(100.0 * v / total, 2)} for k, v in counts.items()}
    print(f"\n[{model_name}] total errors: {total}")
    for cls in ["sign-error", "magnitude-ballpark", "format-garbage", "other"]:
        if cls in counts:
            print(f"  {cls}: {counts[cls]} ({100.0*counts[cls]/total:.1f}%)")

# Persist raw + summary for reproducibility
Path("../results/analysis_v2").mkdir(parents=True, exist_ok=True)
with open("../results/analysis_v2/phase1_raw.json", "w") as f:
    json.dump(all_results, f, indent=2)
with open("../results/analysis_v2/phase1_summary.json", "w") as f:
    json.dump({"op_accuracy": op_accuracy, "failure_histograms": failure_histograms, "error_taxonomy": taxonomy}, f, indent=2)
print("\nPhase 1 results saved to experiments/results/analysis_v2/")

## Phase 2 — Transformer internals (CP1-CP3 successors)

**Protocol note for 2.1/2.2**: attention is extracted from the model's OWN autoregressively-generated sequence (decoded first via free-running greedy decode, then a single forward pass with `return_attention=True` using that generated sequence as decoder input) — no ground-truth answer is used as decoder input. This is genuinely autoregressive-consistent, not teacher-forced, so no protocol banner is needed here (see 2.3 below for the one probe that does need one).

### 2.1 Encoder self-attention maps (CP1 successor) + 2.2 Cross-attention per decoding step (CP2 successor)
ID examples from `val.json`, OOD examples spanning ops4/ops5/ops7.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR = Path("../results/analysis_v2")
(RESULTS_DIR / "attention_maps_v2").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "cross_attention_v2").mkdir(exist_ok=True)
(RESULTS_DIR / "head_ablation_v2").mkdir(exist_ok=True)

def decode_tokens(token_ids):
    return [tokenizer.idx_to_char.get(t, "?") for t in token_ids if t != pad_idx]

def encode_output(text):
    ids = encode_checked(text, MAX_OUTPUT_LEN - 1)  # room for SOS
    return ids

val_data = json.loads((DATA_DIR / "val.json").read_text())["data"]
ood_data = {n: json.loads((DATA_DIR / f"ood_ops{n}.json").read_text())["data"] for n in (4, 5, 6, 7)}

id_examples = val_data[:3]
ood_examples = [ood_data[4][0], ood_data[5][0], ood_data[7][0]]  # one from ops4, ops5, ops7 for spread

lock_on_records = []

def extract_and_plot(item, split_name, idx, quantitative_only=False):
    src = encode_input(item["input"])
    gen_ids = autoregressive_decode(model=transformer, src=src, device=DEVICE)  # includes SOS
    dec_len = len(gen_ids)
    dec_padded = gen_ids + [pad_idx] * (MAX_OUTPUT_LEN - dec_len)
    dec_padded = dec_padded[:MAX_OUTPUT_LEN]
    dec_tensor = torch.tensor([dec_padded], dtype=torch.long).to(DEVICE)
    src_in = src.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits, attn = transformer(src_in, dec_tensor, return_attention=True)

    src_labels = decode_tokens(src.tolist())
    tgt_labels = [tokenizer.idx_to_char.get(t, "?") for t in gen_ids]

    cross_stack = torch.stack([d["cross_attention"][0] for d in attn["decoder_attention"]], dim=0)
    avg_cross = cross_stack.mean(dim=(0, 1)).cpu().numpy()
    s_len = len(src_labels)
    t_len = dec_len
    avg_cross = avg_cross[:t_len, :s_len]

    argmax_per_step = avg_cross.argmax(axis=1).tolist()
    pred_str = tokenizer.decode(gen_ids[1:])
    target_str = str(item["output"])
    lock_on_records.append({
        "split": split_name, "expression": item["input"], "target": target_str, "pred": pred_str,
        "correct": pred_str == target_str, "num_decoding_steps": t_len,
        "argmax_src_token_per_step": argmax_per_step,
        "num_unique_argmax_tokens": len(set(argmax_per_step)),
    })

    if quantitative_only:
        return

    enc_weights = attn["encoder_attention"]
    n_layers = len(enc_weights)
    n_heads = enc_weights[0].shape[1]
    fig, axes = plt.subplots(n_layers, n_heads, figsize=(n_heads * 2, n_layers * 2))
    fig.suptitle(f"CP1-successor Encoder Self-Attention [{split_name} ex{idx+1}]\n{item['input'][:50]} (own-generation decode)", fontsize=9)
    for l in range(n_layers):
        for h in range(n_heads):
            ax = axes[l][h]
            seq_len = len(src_labels)
            attn_map = enc_weights[l][0, h, :seq_len, :seq_len].cpu().numpy()
            sns.heatmap(attn_map, ax=ax, xticklabels=src_labels, yticklabels=src_labels,
                        cmap="Blues", vmin=0, vmax=1, cbar=False, square=True)
            ax.set_title(f"L{l+1}H{h+1}", fontsize=6)
            ax.tick_params(labelsize=4)
    plt.tight_layout()
    path = RESULTS_DIR / f"attention_maps_v2/cp1_{split_name}_ex{idx+1}.png"
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")

    fig, ax = plt.subplots(figsize=(s_len * 0.4 + 2, t_len * 0.35 + 2))
    sns.heatmap(avg_cross, ax=ax, xticklabels=src_labels, yticklabels=tgt_labels,
                cmap="Oranges", vmin=0, vmax=max(avg_cross.max(), 1e-6), cbar=True)
    ax.set_xlabel("Source tokens", fontsize=8)
    ax.set_ylabel("Decoding steps (own generation)", fontsize=8)
    ax.set_title(f"CP2-successor Cross-Attention [{split_name} ex{idx+1}]\n{item['input'][:40]} -> pred={pred_str} (target={target_str})", fontsize=8)
    plt.tight_layout()
    path = RESULTS_DIR / f"cross_attention_v2/cp2_{split_name}_ex{idx+1}.png"
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")

for i, item in enumerate(id_examples):
    extract_and_plot(item, "ID", i)
for i, item in enumerate(ood_examples):
    extract_and_plot(item, "OOD", i)

**Lock-on check (quantitative, larger sample)**: the legacy finding claimed attention "pins to one source token across all decoding steps" on OOD. Tested directly here on the verified checkpoint, not assumed. Metric: for each example, `num_unique_argmax_tokens / num_decoding_steps` — 1.0 means a different source token dominates at every step (fully step-dependent, no lock-on); near-0 means the same token dominates at every step (fully locked-on). Computed over 23 ID examples (val.json[3:23]) and 80 OOD examples (20 each from ops4/5/6/7), all with own-generation decoding, requiring >=2 decoding steps to be meaningful.

In [ ]:
for item in val_data[3:23]:
    extract_and_plot(item, "ID", 0, quantitative_only=True)
for n in (4, 5, 6, 7):
    for item in ood_data[n][1:21]:
        extract_and_plot(item, f"OOD_ops{n}", 0, quantitative_only=True)

with open(RESULTS_DIR / "lock_on_records.json", "w") as f:
    json.dump(lock_on_records, f, indent=2)

lock_on_summary = defaultdict(list)
for r in lock_on_records:
    split_group = "ID" if r["split"] == "ID" else "OOD"
    if r["num_decoding_steps"] >= 2:
        lock_on_summary[split_group].append(r["num_unique_argmax_tokens"] / r["num_decoding_steps"])

print("Lock-on summary (unique-argmax-token-ratio; 1.0=fully varying, near-0=fully locked):")
for grp, vals in lock_on_summary.items():
    if vals:
        print(f"  {grp}: mean={sum(vals)/len(vals):.3f}  n={len(vals)}  min={min(vals):.3f}  max={max(vals):.3f}")

### 2.3 Head ablation (CP3 successor), all 48 heads, absolute + relative drops

**PROTOCOL BANNER**: this probe uses **teacher-forced single-pass evaluation** (ground-truth decoder input fed in one forward call), **not autoregressive generation** — matching the original CP3 methodology exactly. Reason: autoregressive ablation across 48 heads x ~6000 examples x ~11 decode steps each would be computationally prohibitive (~600K+ forward passes vs ~57.6K for teacher-forced). Disclosed per protocol, same as the legacy analysis.

Baseline val is 7.5% (autoregressive), but this teacher-forced ablation population has its own (higher) teacher-forced baselines — absolute drops are small in both regimes, so **relative drops (% of baseline) are reported alongside absolute** to avoid the floor-effect trap of eyeballing tiny absolute numbers.

In [ ]:
def is_correct_teacher_forced(logits, tgt):
    preds = logits.argmax(dim=-1)[0]
    tgt_gt = tgt[1:len(preds)+1]
    eos = tokenizer.eos_idx
    pred_list, gt_list = [], []
    for p, g in zip(preds, tgt_gt):
        pred_list.append(p.item())
        gt_list.append(g.item())
        if g.item() == eos:
            break
    return pred_list == gt_list

def run_forward_tf(model, src, tgt):
    with torch.no_grad():
        tgt_in = tgt[:-1].unsqueeze(0)
        src_in = src.unsqueeze(0)
        logits = model(src_in, tgt_in)
    return logits

def make_full_target(item):
    inp_ids = encode_input(item["input"])
    out_ids = encode_output(str(item["output"]))
    dec = [tokenizer.sos_idx] + out_ids + [tokenizer.eos_idx]
    dec = dec + [pad_idx] * (MAX_OUTPUT_LEN - len(dec))
    dec = dec[:MAX_OUTPUT_LEN]
    return inp_ids, torch.tensor(dec, dtype=torch.long)

id_pop_raw = val_data[:200]
ood_pop_raw = ood_data[4][:250] + ood_data[5][:250] + ood_data[6][:250] + ood_data[7][:250]
print(f"ID population: {len(id_pop_raw)} (val.json[:200])")
print(f"OOD population: {len(ood_pop_raw)} (250 each from ops4/5/6/7, stratified)")

id_pop = [make_full_target(item) for item in id_pop_raw]
ood_pop = [make_full_target(item) for item in ood_pop_raw]

def ablate_and_eval(model, data, layer_idx, head_idx, component):
    correct = 0
    def hook_fn(module, input, output):
        attn_out = output[0].clone()
        d_model = attn_out.shape[-1]
        head_dim = d_model // module.num_heads
        attn_out[:, :, head_idx*head_dim:(head_idx+1)*head_dim] = 0
        return (attn_out,) + output[1:]
    if component == "encoder":
        handle = model.transformer.encoder.layers[layer_idx].self_attn.register_forward_hook(hook_fn)
    else:
        handle = model.transformer.decoder.layers[layer_idx].multihead_attn.register_forward_hook(hook_fn)
    with torch.no_grad():
        for src, tgt in data:
            logits = run_forward_tf(model, src.to(DEVICE), tgt.to(DEVICE))
            if is_correct_teacher_forced(logits, tgt.to(DEVICE)):
                correct += 1
    handle.remove()
    return correct / len(data)

baseline_id = sum(is_correct_teacher_forced(run_forward_tf(transformer, s.to(DEVICE), t.to(DEVICE)), t.to(DEVICE)) for s, t in id_pop) / len(id_pop)
baseline_ood = sum(is_correct_teacher_forced(run_forward_tf(transformer, s.to(DEVICE), t.to(DEVICE)), t.to(DEVICE)) for s, t in ood_pop) / len(ood_pop)
print(f"Baseline ID (teacher-forced): {baseline_id:.4f}")
print(f"Baseline OOD (teacher-forced): {baseline_ood:.4f}")

In [ ]:
results = []
i = 0
for comp in ["encoder", "decoder_cross"]:
    for l in range(3):
        for h in range(8):
            label = f"{comp[0].upper()}L{l+1}H{h+1}"
            acc_id = ablate_and_eval(transformer, id_pop, l, h, comp)
            acc_ood = ablate_and_eval(transformer, ood_pop, l, h, comp)
            id_drop = baseline_id - acc_id
            ood_drop = baseline_ood - acc_ood
            delta = ood_drop - id_drop
            id_drop_rel = (id_drop / baseline_id * 100) if baseline_id > 0 else None
            ood_drop_rel = (ood_drop / baseline_ood * 100) if baseline_ood > 0 else None
            results.append({
                "label": label, "id_drop": id_drop, "ood_drop": ood_drop, "delta": delta,
                "id_drop_rel_pct": id_drop_rel, "ood_drop_rel_pct": ood_drop_rel,
            })
            i += 1
            print(f"  [{i}/48] {label}: id_drop={id_drop:.4f} ood_drop={ood_drop:.4f} delta={delta:.4f}")

with open(RESULTS_DIR / "head_ablation_v2/cp3_v2_results.json", "w") as f:
    json.dump({"baseline_id": baseline_id, "baseline_ood": baseline_ood, "heads": results}, f, indent=2)

top = sorted(results, key=lambda x: x["delta"], reverse=True)[:5]
print("\nTop 5 by delta (absolute OOD-ID drop):")
for r in top:
    print(f"  {r}")

n_cross_001 = sum(1 for r in results if r["delta"] >= 0.01)
print(f"\nHeads with delta >= 0.01 (absolute): {n_cross_001}")
print(f"Max single-head ood_drop: {max(r['ood_drop'] for r in results):.4f}")

**GATE — Phase 2 complete. Hold here.** Phase 3 (LSTM internals + cross-model probing) waits for review of Phase 1 and Phase 2 together.

## Phase 3 — LSTM internals + cross-model probing (new ground)

Load the LSTM (already available as `transformer` from Phase 2; loading `lstm` fresh here with its own fingerprint assertion).

### Ground truth for probing
Position-level targets (running intermediate value, paren-nesting depth) are computed by a small recursive-descent parser over the fully-parenthesized grammar — exact, not approximate. Validated against 200 real examples: the parser's final computed value matched the dataset's own recorded `output` field in all 200 cases.

In [ ]:
import re
import numpy as np
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import r2_score, accuracy_score

lstm = load_lstm()

TOKEN_RE = re.compile(r'\(|\)|\+|\*|-?\d+|/|-')

def compute_position_targets(expr):
    """Exact position-level targets via a small recursive-descent parser over
    the fully-parenthesized grammar. depth_at_char: running paren-nesting depth.
    value_at_char: value of the most recently completed (closed) subexpression,
    forward-filled from each ')' - 0 before the first close."""
    n = len(expr)
    depth_at_char = [0] * n
    depth = 0
    for i, c in enumerate(expr):
        if c == '(':
            depth += 1
        depth_at_char[i] = depth
        if c == ')':
            depth -= 1

    tokens = [(m.group(), m.start(), m.end()) for m in TOKEN_RE.finditer(expr)]
    value_events = []

    def parse(idx):
        tok, start, end = tokens[idx]
        if tok == '(':
            idx += 1
            left_val, idx = parse(idx)
            op_tok, _, _ = tokens[idx]
            idx += 1
            right_val, idx = parse(idx)
            close_tok, close_start, close_end = tokens[idx]
            assert close_tok == ')', f"expected ')' got {close_tok!r} in {expr!r}"
            if op_tok == '+':
                val = left_val + right_val
            elif op_tok == '-':
                val = left_val - right_val
            elif op_tok == '*':
                val = left_val * right_val
            elif op_tok == '/':
                val = left_val // right_val if right_val != 0 else 0
            else:
                raise ValueError(f"unknown op {op_tok!r}")
            value_events.append((close_end - 1, val))
            idx += 1
            return val, idx
        else:
            return int(tok), idx + 1

    parse(0)
    value_events.sort()
    value_at_char = [0] * n
    ei = 0
    current = 0
    for i in range(n):
        while ei < len(value_events) and value_events[ei][0] <= i:
            current = value_events[ei][1]
            ei += 1
        value_at_char[i] = current
    return depth_at_char, value_at_char

# Validate against 200 real examples: parser's final value should match the dataset's own output
_mismatches = 0
for _item in val_data[:200]:
    _, _vt = compute_position_targets(_item["input"])
    if _vt[-1] != int(_item["output"]):
        _mismatches += 1
print(f"Parser validation: {_mismatches} mismatches out of 200 real examples (expect 0)")

### 3.1 LSTM hidden-state probes / 3.2 Transformer encoder probes (mean-pooled across the 3 layers, per position)

Trained on `val.json[:300]` (all real, non-pad character positions), evaluated on a held-out `val.json[300:450]` slice (ID) and 150 examples from each OOD bucket. Ridge regression for the running intermediate value, logistic regression for paren-nesting depth.

In [ ]:
def lstm_encoder_states(model, src):
    with torch.no_grad():
        src_in = src.unsqueeze(0).to(DEVICE)
        outputs, _ = model.encoder(src_in)  # [1, seq, hidden*2]
    return outputs[0].cpu().numpy()

def transformer_encoder_states_mean_pooled(model, src):
    """Manually replicate the encoder-layer loop (same as forward's
    return_attention path) to collect per-layer outputs, mean-pooled across
    the 3 layers per position."""
    with torch.no_grad():
        src_in = src.unsqueeze(0).to(DEVICE)
        src_emb = model.encoder_embedding(src_in) * (model.d_model ** 0.5)
        src_emb = model.pos_encoder(src_emb)
        enc_out = src_emb
        per_layer = []
        for layer in model.transformer.encoder.layers:
            attn_out, _ = layer.self_attn(enc_out, enc_out, enc_out, need_weights=True, average_attn_weights=False)
            enc_out = layer.norm1(enc_out + layer.dropout1(attn_out))
            ff_out = layer.linear2(layer.dropout(layer.activation(layer.linear1(enc_out))))
            enc_out = layer.norm2(enc_out + layer.dropout2(ff_out))
            per_layer.append(enc_out.clone())
        mean_pooled = torch.stack(per_layer, dim=0).mean(dim=0)
    return mean_pooled[0].cpu().numpy()

def build_probe_dataset(model_name, model, items):
    X, y_depth, y_value = [], [], []
    for item in items:
        expr = item["input"]
        depth_targets, value_targets = compute_position_targets(expr)
        src = encode_input(expr)
        states = lstm_encoder_states(model, src) if model_name == "lstm" else transformer_encoder_states_mean_pooled(model, src)
        for pos in range(len(expr)):
            X.append(states[pos])
            y_depth.append(depth_targets[pos])
            y_value.append(value_targets[pos])
    return np.array(X), np.array(y_depth), np.array(y_value)

N_TRAIN, N_ID_TEST, N_OOD_TEST = 300, 150, 150
train_items = val_data[:N_TRAIN]
id_test_items = val_data[N_TRAIN:N_TRAIN + N_ID_TEST]

probe_results = {}
for model_name, model in [("lstm", lstm), ("transformer", transformer)]:
    print(f"\n--- {model_name} ---")
    X_train, y_depth_train, y_value_train = build_probe_dataset(model_name, model, train_items)
    print(f"  train positions: {X_train.shape[0]}, feature dim: {X_train.shape[1]}")

    value_probe = Ridge(alpha=1.0)
    value_probe.fit(X_train, y_value_train)
    depth_probe = LogisticRegression(max_iter=1000)
    depth_probe.fit(X_train, y_depth_train)

    model_results = {}
    X_id, y_depth_id, y_value_id = build_probe_dataset(model_name, model, id_test_items)
    r2_id = r2_score(y_value_id, value_probe.predict(X_id))
    acc_id = accuracy_score(y_depth_id, depth_probe.predict(X_id))
    model_results["ID_val_heldout"] = {"n_positions": len(y_depth_id), "value_r2": r2_id, "depth_acc": acc_id}
    print(f"  ID (val heldout): n={len(y_depth_id)} value_R2={r2_id:.3f} depth_acc={acc_id:.3f}")

    for n in (4, 5, 6, 7):
        X_ood, y_depth_ood, y_value_ood = build_probe_dataset(model_name, model, ood_data[n][:N_OOD_TEST])
        r2_ood = r2_score(y_value_ood, value_probe.predict(X_ood))
        acc_ood = accuracy_score(y_depth_ood, depth_probe.predict(X_ood))
        model_results[f"ops{n}"] = {"n_positions": len(y_depth_ood), "value_r2": r2_ood, "depth_acc": acc_ood}
        print(f"  ops{n}: n={len(y_depth_ood)} value_R2={r2_ood:.3f} depth_acc={acc_ood:.3f}")

    probe_results[model_name] = model_results

with open(RESULTS_DIR / "phase3_probe_results.json", "w") as f:
    json.dump(probe_results, f, indent=2)

In [ ]:
# Depth classification accuracy is meaningless without the trivial majority-class
# baseline for comparison - compute it for ID and every OOD bucket.
from collections import Counter

def majority_class_baseline(items):
    depth_counts = Counter()
    for item in items:
        d, _ = compute_position_targets(item["input"])
        depth_counts.update(d)
    total = sum(depth_counts.values())
    maj_class, maj_n = depth_counts.most_common(1)[0]
    return maj_class, maj_n / total, dict(sorted(depth_counts.items()))

print("Majority-class depth baseline (trivial 'always guess the most common depth'):")
maj_class_id, maj_acc_id, dist_id = majority_class_baseline(id_test_items)
print(f"  ID: majority_class={maj_class_id} baseline_acc={maj_acc_id:.4f} distribution={dist_id}")
ood_baselines = {}
for n in (4, 5, 6, 7):
    maj_class, maj_acc, dist = majority_class_baseline(ood_data[n][:N_OOD_TEST])
    ood_baselines[n] = maj_acc
    print(f"  ops{n}: majority_class={maj_class} baseline_acc={maj_acc:.4f} distribution={dist}")

print("\nProbe depth_acc vs majority-class baseline (probe - baseline; negative = probe loses to trivial guessing):")
for model_name in ["lstm", "transformer"]:
    r = probe_results[model_name]
    print(f"  [{model_name}] ID: {r['ID_val_heldout']['depth_acc']:.3f} - {maj_acc_id:.3f} = {r['ID_val_heldout']['depth_acc']-maj_acc_id:+.3f}")
    for n in (4, 5, 6, 7):
        diff = r[f'ops{n}']['depth_acc'] - ood_baselines[n]
        print(f"  [{model_name}] ops{n}: {r[f'ops{n}']['depth_acc']:.3f} - {ood_baselines[n]:.3f} = {diff:+.3f}")

### 3.3 Per-timestep prediction confidence along generation, both models
Same autoregressive decode as Phase 1, additionally recording the softmax max-probability (confidence) at each step. 200 examples per file.

In [ ]:
def autoregressive_decode_with_confidence(model, src, device, max_len=MAX_OUTPUT_LEN):
    model.eval()
    with torch.no_grad():
        src_in = src.unsqueeze(0).to(device)
        dec_ids = [tokenizer.sos_idx]
        confidences = []
        for _ in range(max_len - 1):
            cur = dec_ids + [pad_idx] * (max_len - len(dec_ids))
            cur = cur[:max_len]
            dec_tensor = torch.tensor([cur], dtype=torch.long).to(device)
            out = model(src_in, dec_tensor)
            probs = torch.softmax(out[0, len(dec_ids) - 1, :], dim=-1)
            conf, nxt = probs.max(dim=-1)
            nxt = nxt.item()
            confidences.append(conf.item())
            if nxt == tokenizer.eos_idx or nxt == pad_idx:
                break
            dec_ids.append(nxt)
    return dec_ids[1:], confidences

N_CONF = 200
conf_results = {}
for model_name, model in [("transformer", transformer), ("lstm", lstm)]:
    step_confidences = {}
    files = {"val": val_data[:N_CONF]}
    for n in (4, 5, 6, 7):
        files[f"ops{n}"] = ood_data[n][:N_CONF]
    for file_key, items in files.items():
        for item in items:
            src = encode_input(item["input"])
            _, confs = autoregressive_decode_with_confidence(model, src, DEVICE)
            for step, c in enumerate(confs):
                step_confidences.setdefault(file_key, {}).setdefault(step, []).append(c)
    conf_results[model_name] = {
        file_key: {str(step): {"mean": float(np.mean(vals)), "n": len(vals)} for step, vals in steps.items()}
        for file_key, steps in step_confidences.items()
    }
    print(f"\n[{model_name}]")
    for file_key in files:
        steps = conf_results[model_name].get(file_key, {})
        line = "  " + file_key + ": " + ", ".join(f"step{s}={steps[s]['mean']:.3f}(n={steps[s]['n']})" for s in sorted(steps, key=int))
        print(line)

with open(RESULTS_DIR / "phase3_confidence_results.json", "w") as f:
    json.dump(conf_results, f, indent=2)

**GATE — Phase 3 complete. Hold here.** No figures go into the paper until all three phases are read together.

## Phase 3 addendum — LSTM cell-state (c_t) probe

`nn.LSTM` only exposes per-position **hidden** states via `outputs`; the `cell` it returns is only the final-timestep state per direction, not one per position. To probe `c_t` at every position, the bidirectional LSTM recurrence is manually unrolled using the trained weights (`weight_ih_l0[_reverse]`, `weight_hh_l0[_reverse]`, biases), then validated by reconstructing the per-position **hidden** states the same way and checking they match `nn.LSTM`'s own `outputs` tensor exactly. All diffs were at float32 precision (~1e-6 to 1e-7), confirming the manual recurrence is correct — including the cell states it also produces via the same computation.

**Combination used**: concatenated forward + backward cell states, aligned per original left-to-right position — identical convention to how `outputs` already concatenates forward/backward hidden states.

In [ ]:
from lstm_manual_unroll import manual_lstm_unroll  # sits next to this notebook

def lstm_cell_states(model, src):
    with torch.no_grad():
        src_in = src.unsqueeze(0).to(DEVICE)
        embedded = model.encoder.dropout(model.encoder.embedding(src_in))
        _, c_fwd, _, c_bwd = manual_lstm_unroll(model.encoder.lstm, embedded)
        c_concat = torch.cat([c_fwd, c_bwd], dim=-1)
    return c_concat[0].cpu().numpy()

# Validation: reconstruct hidden states the same way and confirm they match
# nn.LSTM's own `outputs` exactly - all diffs were float32-precision (~1e-6 to 1e-7).
_src_check = encode_input(val_data[0]["input"])
with torch.no_grad():
    _src_in = _src_check.unsqueeze(0).to(DEVICE)
    _embedded = lstm.encoder.dropout(lstm.encoder.embedding(_src_in))
    _outputs_official, _ = lstm.encoder.lstm(_embedded)
    _h_fwd, _c_fwd, _h_bwd, _c_bwd = manual_lstm_unroll(lstm.encoder.lstm, _embedded)
_h_manual = torch.cat([_h_fwd, _h_bwd], dim=-1)
print("Manual unroll validation - max|manual_h - official_outputs|:", (_h_manual - _outputs_official).abs().max().item())

def build_probe_dataset_statefn(state_fn, items):
    """Generic version of build_probe_dataset that takes an arbitrary
    state-extraction function instead of dispatching on model_name - needed
    here since c_t isn't one of the two cases build_probe_dataset knows about."""
    X, y_depth, y_value = [], [], []
    for item in items:
        expr = item["input"]
        depth_targets, value_targets = compute_position_targets(expr)
        src = encode_input(expr)
        states = state_fn(src)
        for pos in range(len(expr)):
            X.append(states[pos])
            y_depth.append(depth_targets[pos])
            y_value.append(value_targets[pos])
    return np.array(X), np.array(y_depth), np.array(y_value)

In [ ]:
X_train_c, y_depth_train_c, y_value_train_c = build_probe_dataset_statefn(lambda s: lstm_cell_states(lstm, s), train_items)
print(f"train positions: {X_train_c.shape[0]}, feature dim: {X_train_c.shape[1]}")

value_probe_c = Ridge(alpha=1.0)
value_probe_c.fit(X_train_c, y_value_train_c)
depth_probe_c = LogisticRegression(max_iter=1000)
depth_probe_c.fit(X_train_c, y_depth_train_c)

c_results = {}
X_id_c, y_depth_id_c, y_value_id_c = build_probe_dataset_statefn(lambda s: lstm_cell_states(lstm, s), id_test_items)
r2_id_c = r2_score(y_value_id_c, value_probe_c.predict(X_id_c))
acc_id_c = accuracy_score(y_depth_id_c, depth_probe_c.predict(X_id_c))
c_results["ID_val_heldout"] = {"n_positions": len(y_depth_id_c), "value_r2": r2_id_c, "depth_acc": acc_id_c, "depth_majority_baseline": maj_acc_id}
print(f"ID: n={len(y_depth_id_c)} value_R2={r2_id_c:.3f} depth_acc={acc_id_c:.3f} (majority_baseline={maj_acc_id:.3f})")

for n in (4, 5, 6, 7):
    items = ood_data[n][:N_OOD_TEST]
    X_ood_c, y_depth_ood_c, y_value_ood_c = build_probe_dataset_statefn(lambda s: lstm_cell_states(lstm, s), items)
    r2_ood_c = r2_score(y_value_ood_c, value_probe_c.predict(X_ood_c))
    acc_ood_c = accuracy_score(y_depth_ood_c, depth_probe_c.predict(X_ood_c))
    c_results[f"ops{n}"] = {"n_positions": len(y_depth_ood_c), "value_r2": r2_ood_c, "depth_acc": acc_ood_c, "depth_majority_baseline": ood_baselines[n]}
    print(f"ops{n}: n={len(y_depth_ood_c)} value_R2={r2_ood_c:.3f} depth_acc={acc_ood_c:.3f} (majority_baseline={ood_baselines[n]:.3f})")

with open(RESULTS_DIR / "phase3_addendum_celltstate_results.json", "w") as f:
    json.dump(c_results, f, indent=2)

### STEP 3 — Final-timestep-only probes: "is the answer decodable anywhere at the end"

Weaker question than 3.1/3.2's per-position probes (which ask "is the running computation exposed at every step"). Single feature vector per example (the state at the final REAL, non-pad position), target = final answer value. Note the much smaller training set here (300 examples, one vector each, vs. 4372 training positions for the per-position probes) — a higher-variance regression regime, especially for `c_t` which is unbounded (no tanh squashing, unlike `h_t`).

In [ ]:
def build_final_answer_dataset(state_fn, items):
    X, y = [], []
    for item in items:
        expr = item["input"]
        src = encode_input(expr)
        states = state_fn(src)
        final_pos = len(expr) - 1
        X.append(states[final_pos])
        y.append(int(item["output"]))
    return np.array(X), np.array(y)

final_results = {}
probes_to_run = [
    ("lstm_h_final", lambda s: lstm_encoder_states(lstm, s)),
    ("lstm_c_final", lambda s: lstm_cell_states(lstm, s)),
    ("transformer_meanpooled_final", lambda s: transformer_encoder_states_mean_pooled(transformer, s)),
]

for name, state_fn in probes_to_run:
    X_train_f, y_train_f = build_final_answer_dataset(state_fn, train_items)
    probe = Ridge(alpha=1.0)
    probe.fit(X_train_f, y_train_f)

    model_results = {}
    X_id_f, y_id_f = build_final_answer_dataset(state_fn, id_test_items)
    r2_id_f = r2_score(y_id_f, probe.predict(X_id_f))
    model_results["ID_val_heldout"] = {"n_examples": len(y_id_f), "value_r2": r2_id_f}
    print(f"[{name}] ID: n={len(y_id_f)} value_R2={r2_id_f:.3f}")

    for n in (4, 5, 6, 7):
        items = ood_data[n][:N_OOD_TEST]
        X_ood_f, y_ood_f = build_final_answer_dataset(state_fn, items)
        r2_ood_f = r2_score(y_ood_f, probe.predict(X_ood_f))
        model_results[f"ops{n}"] = {"n_examples": len(y_ood_f), "value_r2": r2_ood_f}
        print(f"[{name}] ops{n}: n={len(y_ood_f)} value_R2={r2_ood_f:.3f}")

    final_results[name] = model_results

with open(RESULTS_DIR / "phase3_addendum_final_answer_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

**GATE — Phase 3 addendum complete. Hold here.**